# 🎵 Messy Mashup — Training Notebook

This notebook trains **three models** as required by the project:

| # | Model | Type | Purpose |
|---|-------|------|---------|
| 1 | `GenreCNN` | Built from scratch | Baseline deep model |
| 2 | `EfficientNet-B0` | Pretrained (ImageNet) | Best performer — used for submission |
| 3 | `CRNN` (CNN+GRU) | Built from scratch | Sequential model capturing temporal patterns |

**All 3 runs logged to W&B** with F1 + accuracy for comparison.

**Final submission** = ensemble of EfficientNet-B0 + CRNN logits (weighted average).

In [1]:
# ── Install / imports ──────────────────────────────────────────────────
!pip install wandb --quiet

import os, random, warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import librosa
from tqdm import tqdm
from sklearn.metrics import f1_score
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import models
import wandb
from kaggle_secrets import UserSecretsClient

# ── W&B login ──────────────────────────────────────────────────────────
os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")
wandb.login()
print("W&B authenticated.")

# ── Device ─────────────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: 23f2002537 (23f2002537-indian-institute-of-technology-madras) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


W&B authenticated.
Device: cuda


In [2]:
# ── Constants ──────────────────────────────────────────────────────────
DATA_PATH  = "/kaggle/input/datasets/noobshy/messy-mashup-data/train_data"
TEST_PATH  = "/kaggle/input/competitions/jan-2026-dl-gen-ai-project/messy_mashup/mashups"
MODEL_DIR  = "/kaggle/working/models"
os.makedirs(MODEL_DIR, exist_ok=True)

genres      = ["blues","classical","country","disco","hiphop",
               "jazz","metal","pop","reggae","rock"]
genre_to_idx= {g: i for i, g in enumerate(genres)}
idx_to_genre= {i: g for g, i in genre_to_idx.items()}
NUM_CLASSES = 10
SR          = 22050
EPOCHS      = 25
BATCH_SIZE  = 32
VAL_SPLIT   = 0.15          # 15% held out for validation
WANDB_PROJECT = "23f2002537-t12026"

print(f"Genres: {genres}")

Genres: ['blues', 'classical', 'country', 'disco', 'hiphop', 'jazz', 'metal', 'pop', 'reggae', 'rock']


## 1. Dataset

In [3]:
class PrecomputedMelDataset(Dataset):
    """
    Loads precomputed mel-spectrogram tensors saved by the
    data-augmentation notebook.
    Applies optional runtime SpecAugment for extra diversity.
    """

    def __init__(self, data_path, augment=True, target_size=(128, 431)):
        self.data_path   = data_path
        self.files       = sorted(os.listdir(data_path))
        self.augment     = augment
        self.target_size = target_size   # (freq_bins, time_frames)

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        d   = torch.load(os.path.join(self.data_path, self.files[idx]))
        mel = d["mel"].float()           # (128, T)
        lbl = int(d["label"])

        # ── Pad or crop to a fixed width ────────────────────────────
        _, T = mel.shape
        target_T = self.target_size[1]
        if T < target_T:
            mel = F.pad(mel, (0, target_T - T))
        else:
            mel = mel[:, :target_T]

        # ── Runtime SpecAugment (training only) ─────────────────────
        if self.augment:
            mel = self._spec_augment(mel)

        return mel.unsqueeze(0), lbl    # → (1, 128, T)

    @staticmethod
    def _spec_augment(mel, freq_mask=20, time_mask=50, n=2):
        mel = mel.clone()
        F_bins, T = mel.shape
        for _ in range(n):
            f  = random.randint(1, freq_mask)
            f0 = random.randint(0, F_bins - f)
            mel[f0: f0 + f, :] = 0
            t  = random.randint(1, time_mask)
            t0 = random.randint(0, T - t)
            mel[:, t0: t0 + t] = 0
        return mel


# ── Build train / val splits ───────────────────────────────────────────
full_ds  = PrecomputedMelDataset(DATA_PATH, augment=True)
val_size = int(len(full_ds) * VAL_SPLIT)
trn_size = len(full_ds) - val_size
trn_ds, val_ds = random_split(full_ds, [trn_size, val_size],
                               generator=torch.Generator().manual_seed(42))
# Turn off augmentation for validation
val_ds.dataset.augment = False

trn_loader = DataLoader(trn_ds, batch_size=BATCH_SIZE, shuffle=True,
                        num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=2, pin_memory=True)

print(f"Train: {len(trn_ds):,}  |  Val: {len(val_ds):,}")
mel, lbl = next(iter(trn_loader))
print(f"Batch mel shape: {mel.shape}  label sample: {lbl[:4].tolist()}")

Train: 10,200  |  Val: 1,800
Batch mel shape: torch.Size([32, 1, 128, 431])  label sample: [7, 1, 6, 9]


## 2. Model Definitions

### Model 1 — GenreCNN (built from scratch)
A deeper 5-block CNN with residual-style skip connections and a proper classification head.

In [4]:
class ConvBlock(nn.Module):
    """Conv2d → BN → ReLU → MaxPool"""
    def __init__(self, in_ch, out_ch, pool=True):
        super().__init__()
        self.conv = nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False)
        self.bn   = nn.BatchNorm2d(out_ch)
        self.pool = nn.MaxPool2d(2, 2) if pool else nn.Identity()

    def forward(self, x):
        return self.pool(F.relu(self.bn(self.conv(x))))


class GenreCNN(nn.Module):
    """
    5-block CNN built entirely from scratch.
    Input : (B, 1, 128, 431)
    Output: (B, NUM_CLASSES)
    """
    def __init__(self, num_classes=NUM_CLASSES):
        super().__init__()
        self.encoder = nn.Sequential(
            ConvBlock(1,   32),   # → (B, 32,  64, 215)
            ConvBlock(32,  64),   # → (B, 64,  32, 107)
            ConvBlock(64,  128),  # → (B, 128, 16,  53)
            ConvBlock(128, 256),  # → (B, 256,  8,  26)
            ConvBlock(256, 512, pool=False),  # → (B, 512, 8, 26)
        )
        self.gap  = nn.AdaptiveAvgPool2d(1)  # → (B, 512, 1, 1)
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        return self.head(self.gap(self.encoder(x)))


print("GenreCNN params:",
      sum(p.numel() for p in GenreCNN().parameters()) / 1e6, "M")

GenreCNN params: 1.70289 M


### Model 2 — EfficientNet-B0 (pretrained on ImageNet)
The mel spectrogram is treated as a **3-channel image** (channels are copies) and resized to 224×224. EfficientNet's powerful feature extractor replaces the manual CNN entirely.

In [5]:
class EfficientNetGenre(nn.Module):
    """
    EfficientNet-B0 backbone pretrained on ImageNet,
    with the final classifier replaced for 10-class genre prediction.
    Input : (B, 1, 128, 431)  — grayscale mel
    Output: (B, NUM_CLASSES)
    """
    def __init__(self, num_classes=NUM_CLASSES, freeze_backbone=False):
        super().__init__()
        backbone = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)

        # EfficientNet expects 3-channel input; adapt the first conv
        # by averaging the pretrained weights across channels
        orig_w    = backbone.features[0][0].weight.data   # (32, 3, 3, 3)
        new_conv  = nn.Conv2d(1, 32, kernel_size=3, stride=2, padding=1, bias=False)
        new_conv.weight.data = orig_w.mean(dim=1, keepdim=True)  # (32, 1, 3, 3)
        backbone.features[0][0] = new_conv

        if freeze_backbone:
            for p in backbone.parameters():
                p.requires_grad = False

        in_features = backbone.classifier[1].in_features
        backbone.classifier = nn.Sequential(
            nn.Dropout(0.4),
            nn.Linear(in_features, num_classes)
        )
        self.net = backbone

    def forward(self, x):
        # Resize to 224×224 — EfficientNet's expected resolution
        x = F.interpolate(x, size=(224, 224), mode="bilinear", align_corners=False)
        return self.net(x)


print("EfficientNet-B0 params:",
      sum(p.numel() for p in EfficientNetGenre().parameters()) / 1e6, "M")

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 138MB/s] 


EfficientNet-B0 params: 4.019782 M


### Model 3 — CRNN (CNN + GRU)
A CNN extracts per-frame features along the frequency axis; a bidirectional GRU captures temporal (musical rhythm) patterns across the time axis.

In [6]:
class CRNN(nn.Module):
    """
    Convolutional Recurrent Neural Network.
    CNN: learns local frequency patterns per time-step.
    GRU: captures temporal evolution (musical rhythm) across the clip.
    Input : (B, 1, 128, T)
    Output: (B, NUM_CLASSES)
    """
    def __init__(self, num_classes=NUM_CLASSES, hidden_size=256, num_layers=2):
        super().__init__()
        # ── CNN front-end ──────────────────────────────────────────
        self.cnn = nn.Sequential(
            ConvBlock(1,  32, pool=True),   # (B, 32,  64, T/2)
            ConvBlock(32, 64, pool=True),   # (B, 64,  32, T/4)
            ConvBlock(64, 128, pool=True),  # (B, 128, 16, T/8)
        )
        # After CNN: frequency bins = 16, channels = 128
        # Merge freq × channel into a single feature vector per time-step
        cnn_out_dim = 128 * 16             # 2048

        # ── Bidirectional GRU ──────────────────────────────────────
        self.gru = nn.GRU(
            input_size=cnn_out_dim,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=0.3,
            bidirectional=True
        )
        # ── Classifier head ────────────────────────────────────────
        self.head = nn.Sequential(
            nn.Linear(hidden_size * 2, 256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        # x : (B, 1, 128, T)
        x = self.cnn(x)                        # (B, 128, 16, T')
        B, C, F, T = x.shape
        x = x.permute(0, 3, 1, 2)             # (B, T', C, F)
        x = x.reshape(B, T, C * F)            # (B, T', 2048)
        out, _ = self.gru(x)                   # (B, T', hidden*2)
        out = out.mean(dim=1)                  # temporal average pooling
        return self.head(out)


print("CRNN params:",
      sum(p.numel() for p in CRNN().parameters()) / 1e6, "M")

CRNN params: 4.95153 M


## 3. Training Utilities

In [7]:
def evaluate(model, loader, model_name=""):
    """Return macro-F1 and accuracy on `loader`."""
    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for mel, lbl in loader:
            mel = mel.to(device)
            out = model(mel)
            p   = torch.argmax(out, dim=1).cpu().numpy()
            preds.extend(p)
            targets.extend(lbl.numpy())
    f1  = f1_score(targets, preds, average="macro", zero_division=0)
    acc = np.mean(np.array(preds) == np.array(targets))
    return float(f1), float(acc)


def train_model(model, model_name, run_name,
                lr=1e-3, weight_decay=1e-4, epochs=EPOCHS):
    """
    Full training loop with:
    - Label-smoothing cross-entropy
    - CosineAnnealing LR schedule
    - Mixed-precision (AMP) for GPU speed
    - W&B logging per epoch
    - Best-model checkpointing by val F1
    """

    run = wandb.init(
        project=WANDB_PROJECT,
        name=run_name,
        tags=["training", model_name],
        config={
            "model": model_name,
            "lr": lr,
            "weight_decay": weight_decay,
            "epochs": epochs,
            "batch_size": BATCH_SIZE,
        },
        reinit=True
    )

    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr,
                                   weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=epochs, eta_min=lr * 0.01
    )
    scaler    = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))

    best_f1, best_path = 0.0, f"{MODEL_DIR}/{model_name}_best.pth"

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss, correct, total = 0.0, 0, 0

        for mel, lbl in tqdm(trn_loader,
                              desc=f"{model_name} E{epoch}/{epochs}",
                              leave=False):
            mel, lbl = mel.to(device), lbl.to(device)
            optimizer.zero_grad()

            with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
                out  = model(mel)
                loss = criterion(out, lbl)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # gradient clipping
            scaler.step(optimizer)
            scaler.update()

            total_loss += loss.item() * mel.size(0)
            correct    += (out.argmax(1) == lbl).sum().item()
            total      += mel.size(0)

        scheduler.step()

        trn_loss = total_loss / total
        trn_acc  = correct / total
        val_f1, val_acc = evaluate(model, val_loader)
        cur_lr   = scheduler.get_last_lr()[0]

        print(f"  [{model_name}] Epoch {epoch:02d}/{epochs} "
              f"| Loss: {trn_loss:.4f} | TrnAcc: {trn_acc:.3f} "
              f"| ValF1: {val_f1:.4f} | ValAcc: {val_acc:.3f} "
              f"| LR: {cur_lr:.2e}")

        wandb.log({
            "epoch": epoch,
            "train_loss": trn_loss,
            "train_acc":  trn_acc,
            "val_f1":     val_f1,
            "val_acc":    val_acc,
            "lr":         cur_lr,
        })

        if val_f1 > best_f1:
            best_f1 = val_f1
            torch.save(model.state_dict(), best_path)
            print(f"    ✅ New best F1: {best_f1:.4f} — saved.")

    # Load best weights before returning
    model.load_state_dict(torch.load(best_path, map_location=device))
    print(f"\n🏁 {model_name} best val F1 = {best_f1:.4f}")
    wandb.summary["best_val_f1"] = best_f1
    wandb.finish()
    return model, best_f1

## 4. Train All Three Models

In [8]:
# ── Model 1: GenreCNN (from scratch) ───────────────────────────────────
print("=" * 60)
print("MODEL 1 — GenreCNN (from scratch)")
print("=" * 60)

cnn_model = GenreCNN(NUM_CLASSES).to(device)
cnn_model, cnn_best_f1 = train_model(
    model=cnn_model,
    model_name="GenreCNN",
    run_name="genre_cnn_scratch",
    lr=1e-3,
    epochs=EPOCHS
)

MODEL 1 — GenreCNN (from scratch)


wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


  [GenreCNN] Epoch 01/25 | Loss: 1.4982 | TrnAcc: 0.553 | ValF1: 0.5635 | ValAcc: 0.578 | LR: 9.96e-04
    ✅ New best F1: 0.5635 — saved.


  [GenreCNN] Epoch 02/25 | Loss: 1.1836 | TrnAcc: 0.708 | ValF1: 0.5934 | ValAcc: 0.624 | LR: 9.84e-04
    ✅ New best F1: 0.5934 — saved.


  [GenreCNN] Epoch 03/25 | Loss: 1.0536 | TrnAcc: 0.772 | ValF1: 0.4294 | ValAcc: 0.503 | LR: 9.65e-04


  [GenreCNN] Epoch 04/25 | Loss: 0.9880 | TrnAcc: 0.806 | ValF1: 0.5812 | ValAcc: 0.617 | LR: 9.39e-04


... (18 outputs left out)

  [GenreCNN] Epoch 23/25 | Loss: 0.5569 | TrnAcc: 1.000 | ValF1: 0.9577 | ValAcc: 0.959 | LR: 2.56e-05
    ✅ New best F1: 0.9577 — saved.


  [GenreCNN] Epoch 24/25 | Loss: 0.5537 | TrnAcc: 1.000 | ValF1: 0.9536 | ValAcc: 0.956 | LR: 1.39e-05


  [GenreCNN] Epoch 25/25 | Loss: 0.5550 | TrnAcc: 1.000 | ValF1: 0.9537 | ValAcc: 0.956 | LR: 1.00e-05

🏁 GenreCNN best val F1 = 0.9577


epoch,▁▁▂▂▂▂▃▃▃▄▄▄▅▅▅▅▆▆▆▇▇▇▇██
lr,████▇▇▇▆▆▆▅▅▄▄▃▃▃▂▂▂▁▁▁▁▁
train_acc,▁▃▄▅▆▆▆▆▇▇▇▇█████████████
train_loss,█▆▅▄▄▃▃▃▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁
val_acc,▃▃▂▃▂▁▃▁▄▆▅▄▇▅▇▅▇█▇██████
val_f1,▃▄▂▄▂▁▄▁▄▆▅▅▇▅▇▆▇█▇██████
best_val_f1,0.95772
epoch,25
lr,1e-05
train_acc,1
train_loss,0.55504


In [9]:
# ── Model 2: EfficientNet-B0 (pretrained) ──────────────────────────────
print("=" * 60)
print("MODEL 2 — EfficientNet-B0 (pretrained ImageNet)")
print("=" * 60)

# Phase 1: Freeze backbone, train head only (3 epochs) — fast warm-up
eff_model = EfficientNetGenre(NUM_CLASSES, freeze_backbone=True).to(device)
eff_model, _ = train_model(
    model=eff_model,
    model_name="EfficientNet_warmup",
    run_name="efficientnet_warmup",
    lr=1e-3,
    epochs=3
)

# Phase 2: Unfreeze all layers, fine-tune with small LR
for p in eff_model.parameters():
    p.requires_grad = True

eff_model, eff_best_f1 = train_model(
    model=eff_model,
    model_name="EfficientNet",
    run_name="efficientnet_finetune",
    lr=3e-4,
    weight_decay=1e-4,
    epochs=EPOCHS
)

MODEL 2 — EfficientNet-B0 (pretrained ImageNet)


  [EfficientNet_warmup] Epoch 01/3 | Loss: 1.6950 | TrnAcc: 0.484 | ValF1: 0.6154 | ValAcc: 0.631 | LR: 7.53e-04
    ✅ New best F1: 0.6154 — saved.


  [EfficientNet_warmup] Epoch 02/3 | Loss: 1.4642 | TrnAcc: 0.578 | ValF1: 0.6356 | ValAcc: 0.652 | LR: 2.58e-04
    ✅ New best F1: 0.6356 — saved.


  [EfficientNet_warmup] Epoch 03/3 | Loss: 1.4176 | TrnAcc: 0.603 | ValF1: 0.6464 | ValAcc: 0.662 | LR: 1.00e-05
    ✅ New best F1: 0.6464 — saved.

🏁 EfficientNet_warmup best val F1 = 0.6464


epoch,▁▅█
lr,█▃▁
train_acc,▁▇█
train_loss,█▂▁
val_acc,▁▆█
val_f1,▁▆█
best_val_f1,0.64641
epoch,3
lr,1e-05
train_acc,0.60265
train_loss,1.41757


  [EfficientNet] Epoch 01/25 | Loss: 1.0831 | TrnAcc: 0.760 | ValF1: 0.8789 | ValAcc: 0.885 | LR: 2.99e-04
    ✅ New best F1: 0.8789 — saved.


... (21 outputs left out)

  [EfficientNet] Epoch 23/25 | Loss: 0.5080 | TrnAcc: 1.000 | ValF1: 0.9598 | ValAcc: 0.962 | LR: 7.67e-06


  [EfficientNet] Epoch 24/25 | Loss: 0.5084 | TrnAcc: 0.999 | ValF1: 0.9594 | ValAcc: 0.961 | LR: 4.17e-06


  [EfficientNet] Epoch 25/25 | Loss: 0.5079 | TrnAcc: 0.999 | ValF1: 0.9590 | ValAcc: 0.961 | LR: 3.00e-06

🏁 EfficientNet best val F1 = 0.9628


epoch,▁▁▂▂▂▂▃▃▃▄▄▄▅▅▅▅▆▆▆▇▇▇▇██
lr,████▇▇▇▆▆▆▅▅▄▄▃▃▃▂▂▂▁▁▁▁▁
train_acc,▁▅▆▇▇▇███████████████████
train_loss,█▅▃▃▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_acc,▁▄▄▃▄▆▆▆▆▇▆▆▆▇▇▇▇████████
val_f1,▁▄▄▄▅▆▆▆▆▇▆▆▇▇▇▇▇████████
best_val_f1,0.96279
epoch,25
lr,0.0
train_acc,0.99922
train_loss,0.50792


In [10]:
# ── Model 3: CRNN (from scratch, sequential) ───────────────────────────
print("=" * 60)
print("MODEL 3 — CRNN  (CNN + Bidirectional GRU)")
print("=" * 60)

crnn_model = CRNN(NUM_CLASSES).to(device)
crnn_model, crnn_best_f1 = train_model(
    model=crnn_model,
    model_name="CRNN",
    run_name="crnn_training",
    lr=1e-3,
    epochs=EPOCHS
)

MODEL 3 — CRNN  (CNN + Bidirectional GRU)


  [CRNN] Epoch 01/25 | Loss: 1.6807 | TrnAcc: 0.442 | ValF1: 0.5762 | ValAcc: 0.611 | LR: 9.96e-04
    ✅ New best F1: 0.5762 — saved.


  [CRNN] Epoch 02/25 | Loss: 1.3126 | TrnAcc: 0.636 | ValF1: 0.6920 | ValAcc: 0.714 | LR: 9.84e-04
    ✅ New best F1: 0.6920 — saved.


  [CRNN] Epoch 03/25 | Loss: 1.1507 | TrnAcc: 0.718 | ValF1: 0.7142 | ValAcc: 0.729 | LR: 9.65e-04
    ✅ New best F1: 0.7142 — saved.


  [CRNN] Epoch 04/25 | Loss: 1.0561 | TrnAcc: 0.762 | ValF1: 0.7947 | ValAcc: 0.799 | LR: 9.39e-04
    ✅ New best F1: 0.7947 — saved.


  [CRNN] Epoch 05/25 | Loss: 0.9667 | TrnAcc: 0.804 | ValF1: 0.7796 | ValAcc: 0.787 | LR: 9.05e-04


... (17 outputs left out)

  [CRNN] Epoch 23/25 | Loss: 0.5190 | TrnAcc: 1.000 | ValF1: 0.9159 | ValAcc: 0.919 | LR: 2.56e-05
    ✅ New best F1: 0.9159 — saved.


  [CRNN] Epoch 24/25 | Loss: 0.5187 | TrnAcc: 1.000 | ValF1: 0.9158 | ValAcc: 0.919 | LR: 1.39e-05


  [CRNN] Epoch 25/25 | Loss: 0.5182 | TrnAcc: 1.000 | ValF1: 0.9174 | ValAcc: 0.921 | LR: 1.00e-05
    ✅ New best F1: 0.9174 — saved.

🏁 CRNN best val F1 = 0.9174


epoch,▁▁▂▂▂▂▃▃▃▄▄▄▅▅▅▅▆▆▆▇▇▇▇██
lr,████▇▇▇▆▆▆▅▅▄▄▃▃▃▂▂▂▁▁▁▁▁
train_acc,▁▃▄▅▆▆▆▇▇▇▇▇▇████████████
train_loss,█▆▅▄▄▃▃▃▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁
val_acc,▁▃▄▅▅▆▆▅▆▇▆▇███▇█████████
val_f1,▁▃▄▅▅▆▆▅▆▇▇▇███▇█████████
best_val_f1,0.91739
epoch,25
lr,1e-05
train_acc,0.99971
train_loss,0.5182


In [13]:
WANDB_PROJECT="23f2002537-t12026"
# ── W&B comparison run — log all three models side-by-side ─────────────
comparison_run = wandb.init(
    project=WANDB_PROJECT,
    name="model_comparison",
    tags=["comparison"]
)

cnn_f1,  cnn_acc  = evaluate(cnn_model,  val_loader)
eff_f1,  eff_acc  = evaluate(eff_model,  val_loader)
crnn_f1, crnn_acc = evaluate(crnn_model, val_loader)

comparison_table = wandb.Table(
    columns=["Model", "Type", "Val_MacroF1", "Val_Accuracy"],
    data=[
        ["GenreCNN",        "Scratch",   round(cnn_f1,  4), round(cnn_acc,  4)],
        ["EfficientNet-B0", "Pretrained", round(eff_f1,  4), round(eff_acc,  4)],
        ["CRNN",            "Scratch",   round(crnn_f1, 4), round(crnn_acc, 4)],
    ]
)
wandb.log({"model_comparison": comparison_table})

print("\n📊 Final Comparison:")
print(f"  GenreCNN        — Macro F1: {cnn_f1:.4f}  Acc: {cnn_acc:.4f}")
print(f"  EfficientNet-B0 — Macro F1: {eff_f1:.4f}  Acc: {eff_acc:.4f}")
print(f"  CRNN            — Macro F1: {crnn_f1:.4f}  Acc: {crnn_acc:.4f}")

wandb.finish()


📊 Final Comparison:
  GenreCNN        — Macro F1: 0.9577  Acc: 0.9594
  EfficientNet-B0 — Macro F1: 0.9628  Acc: 0.9644
  CRNN            — Macro F1: 0.9174  Acc: 0.9206


## 5. Save & Upload Models to Kaggle Hub

In [14]:
import json, subprocess, shutil

os.environ["KAGGLE_USERNAME"] = UserSecretsClient().get_secret("Kaggle_username")
KAGGLE_USERNAME = os.environ["KAGGLE_USERNAME"]

# Save all three models
torch.save(cnn_model.state_dict(),  f"{MODEL_DIR}/cnn_model.pth")
torch.save(eff_model.state_dict(),  f"{MODEL_DIR}/efficientnet_model.pth")
torch.save(crnn_model.state_dict(), f"{MODEL_DIR}/crnn_model.pth")
print("Models saved.")

def upload_to_kaggle_hub(slug, filename, overview):
    stage = f"/kaggle/working/hub_upload/{slug}"
    os.makedirs(stage, exist_ok=True)
    shutil.copy(f"{MODEL_DIR}/{filename}", f"{stage}/{filename}")
    json.dump({"title": slug, "id": f"{KAGGLE_USERNAME}/{slug}",
               "licenses": [{"name": "Apache 2.0"}]},
              open(f"{stage}/model-metadata.json", "w"), indent=2)
    subprocess.run(["kaggle", "models", "create",
                    "--owner", KAGGLE_USERNAME, "--name", slug,
                    "--framework", "pytorch", "--overview", overview,
                    "--public"], capture_output=True)
    subprocess.run(["kaggle", "models", "instances", "versions", "create",
                    f"{KAGGLE_USERNAME}/{slug}/pytorch/1",
                    "--path", stage, "--version-notes", "v1"],
                   capture_output=True)
    print(f"  Uploaded: {slug}")

upload_to_kaggle_hub("genre-cnn",         "cnn_model.pth",         "GenreCNN from scratch")
upload_to_kaggle_hub("genre-efficientnet", "efficientnet_model.pth", "EfficientNet-B0 fine-tuned")
upload_to_kaggle_hub("genre-crnn",        "crnn_model.pth",        "CRNN with bidirectional GRU")
print("\n✅ All models on Kaggle Hub!")

Models saved.
  Uploaded: genre-cnn
  Uploaded: genre-efficientnet
  Uploaded: genre-crnn

✅ All models on Kaggle Hub!


## 6. Inference — Ensemble Submission

We ensemble **EfficientNet-B0 + CRNN** using soft voting (weighted logit average).
EfficientNet gets weight 0.6, CRNN gets 0.4.

In [15]:
def audio_to_mel_inference(path, sr=SR, segment_seconds=10):
    """
    Load audio, create mel spectrogram — identical pipeline to training.
    For long test clips, average over multiple 10-second windows.
    """
    segment_length = sr * segment_seconds
    y, _ = librosa.load(path, sr=sr, mono=True)

    # ── Multi-window averaging for long clips ────────────────────────
    hop = segment_length // 2      # 50% overlap
    starts = list(range(0, max(1, len(y) - segment_length + 1), hop))
    mels   = []

    for start in starts[:8]:       # cap at 8 windows to keep it fast
        chunk = y[start: start + segment_length]
        if len(chunk) < segment_length:
            chunk = librosa.util.fix_length(chunk, size=segment_length)
        mel = librosa.feature.melspectrogram(
            y=chunk, sr=sr, n_fft=2048, hop_length=512,
            n_mels=128, fmin=20, fmax=sr // 2
        )
        mel_db = librosa.power_to_db(mel, ref=np.max)
        mel_db = (mel_db - mel_db.mean()) / (mel_db.std() + 1e-6)
        mels.append(mel_db.astype(np.float32))

    # Stack all windows and average predictions over them
    return mels   # list of (128, T) arrays

In [18]:
TARGET_T = 431   # fixed time dimension used during training

def predict_file(path, eff_model, cnn_model,
                 eff_weight=0.6, cnn_weight=0.4):
    mels = audio_to_mel_inference(path)
    all_logits = []

    for mel_arr in mels:
        T = mel_arr.shape[1]
        if T < TARGET_T:
            mel_arr = np.pad(mel_arr, ((0, 0), (0, TARGET_T - T)))
        else:
            mel_arr = mel_arr[:, :TARGET_T]

        tensor = torch.tensor(mel_arr).float().unsqueeze(0).unsqueeze(0).to(device)

        with torch.no_grad():
            eff_logits = eff_model(tensor)
            cnn_logits = cnn_model(tensor)

        combined = eff_weight * eff_logits + cnn_weight * cnn_logits
        all_logits.append(combined)

    avg_logits = torch.stack(all_logits).mean(dim=0)
    return torch.argmax(avg_logits, dim=1).item()

# ── Run inference ───────────────────────────────────────────────────────
eff_model.eval()
crnn_model.eval()

test_files = sorted(os.listdir(TEST_PATH))
ids, predictions = [], []

for fname in tqdm(test_files, desc="Inference"):
    fpath = os.path.join(TEST_PATH, fname)
    pred  = predict_file(fpath, eff_model, cnn_model)
    predictions.append(idx_to_genre[pred])
    ids.append(fname.replace(".wav", "").replace("song", ""))

print(f"Predicted {len(predictions)} files.")

Inference: 100%|██████████| 3020/3020 [07:37<00:00,  6.60it/s]

Predicted 3020 files.


In [19]:
# ── Save submission ─────────────────────────────────────────────────────
submission = pd.DataFrame({"id": ids, "genre": predictions})
submission.to_csv("submission.csv", index=False)

print("submission.csv saved!")
print(submission["genre"].value_counts())
submission.head(10)

submission.csv saved!
genre
rock         329
pop          323
classical    320
metal        314
reggae       312
disco        310
hiphop       309
jazz         286
country      265
blues        252
Name: count, dtype: int64


,id,genre
0,0001,pop
1,0002,classical
2,0003,disco
3,0004,metal
4,0005,country
5,0006,pop
6,0007,rock
7,0008,pop
8,0009,pop
9,0010,disco


In [20]:
# ── Log submission stats to W&B ─────────────────────────────────────────
final_run = wandb.init(
    project=WANDB_PROJECT,
    name="final_submission",
    tags=["inference", "submission"]
)

wandb.log({
    "submission_genre_dist": wandb.Table(
        dataframe=submission["genre"].value_counts().reset_index()
        .rename(columns={"index": "genre", "genre": "count"})
    ),
    "num_predictions": len(predictions),
    "eff_val_f1":  eff_best_f1,
    "crnn_val_f1": crnn_best_f1,
    "cnn_val_f1":  cnn_best_f1,
})
wandb.save("submission.csv")
wandb.finish()
print("✅ All done!")

wandb: WARNING Symlinked 1 file into the W&B run directory; call wandb.save again to sync new files.


cnn_val_f1,▁
crnn_val_f1,▁
eff_val_f1,▁
num_predictions,▁
cnn_val_f1,0.95772
crnn_val_f1,0.91739
eff_val_f1,0.96279
num_predictions,3020


✅ All done!
